# 02 · Embed — `scale_100K` (100,000 papers)
BGE-M3 dense + BM25 index. Platform: Kaggle T4 — only if compute available

In [ ]:
# ══════════════════════════════════════════════════════════════
# SCALE EXPERIMENT CONFIG  ← only N_PAPERS and SCALE_LABEL change between tiers
# ══════════════════════════════════════════════════════════════
import os, sys, json, random
import numpy as np
import pandas as pd
import torch

SCALE_LABEL   = "scale_100K"        # identifier used in output filenames
N_PAPERS      = 100000              # ← THE ONLY NUMBER THAT CHANGES PER TIER
RANDOM_SEED   = 42               # NEVER change this
CHUNK_SIZE    = 400              # tokens
CHUNK_OVERLAP = 50               # tokens
RRF_K         = 60               # standard RRF parameter
TOP_K_STAGE1  = 50               # candidates sent to reranker
EVAL_K_VALUES = [1, 3, 5, 10, 20]

# ── Hardware (fixed by FIX 1) ─────────────────────────────────────────────────
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_ROOT    = "../../1_data"
RESULTS_DIR  = f"../../4_results/{SCALE_LABEL}"
GT_PATH      = f"{DATA_ROOT}/eval/ground_truth.json"
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Scale : {SCALE_LABEL} | N = {N_PAPERS:,} | Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU   : {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

In [ ]:
import time
import chromadb
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import pickle

# Load chunks
df_chunks = pd.read_parquet(f"{RESULTS_DIR}/chunks.parquet")
texts = df_chunks['text'].tolist()
chunk_ids = df_chunks['chunk_id'].tolist()
print(f"Chunks to embed: {len(texts):,}")

In [ ]:
# ── BGE-M3 Dense Embeddings ───────────────────────────────────────────────────
print("Loading BGE-M3...")
model = SentenceTransformer("BAAI/bge-m3", device=DEVICE)

BATCH_SIZE = 64 if DEVICE == 'cuda' else 8
print(f"Embedding {len(texts):,} chunks at batch={BATCH_SIZE}...")
t0 = time.time()
embeddings = model.encode(texts, batch_size=BATCH_SIZE, show_progress_bar=True,
                           normalize_embeddings=True)
elapsed = time.time() - t0
print(f"Done in {elapsed/60:.1f} min | Shape: {embeddings.shape}")

In [ ]:
# ── Save embeddings + ChromaDB index ─────────────────────────────────────────
import numpy as np
emb_path = f"{RESULTS_DIR}/bge_m3_embeddings.npy"
np.save(emb_path, embeddings)
print(f"Saved embeddings: {emb_path}")

client = chromadb.PersistentClient(path=f"{RESULTS_DIR}/chroma_db")
try:
    client.delete_collection(SCALE_LABEL)
except:
    pass
collection = client.create_collection(SCALE_LABEL, metadata={"hnsw:space": "cosine"})
collection.add(
    embeddings=embeddings.tolist(),
    documents=texts,
    ids=chunk_ids,
    metadatas=[{"cord_uid": df_chunks.iloc[i]['cord_uid']} for i in range(len(texts))]
)
print(f"ChromaDB collection '{SCALE_LABEL}': {collection.count()} docs")

In [ ]:
# ── BM25 Index ────────────────────────────────────────────────────────────────
tokenized = [t.lower().split() for t in texts]
bm25 = BM25Okapi(tokenized)
bm25_path = f"{RESULTS_DIR}/bm25_index.pkl"
with open(bm25_path, "wb") as f:
    pickle.dump(bm25, f)
print(f"BM25 index saved: {bm25_path}")

In [ ]:
# ── Semantic sanity check ─────────────────────────────────────────────────────
sanity_pairs = [
    ("CT scans COVID-19 lung pneumonia",
     "COVID-19 lung imaging ground glass opacity",
     "Quantum computing error correction"),
    ("ACE2 receptor SARS-CoV-2 spike protein binding",
     "spike protein ACE2 binding affinity",
     "Recipe for chocolate cake"),
]
from sklearn.metrics.pairwise import cosine_similarity
q_embs = model.encode([p[0] for p in sanity_pairs], normalize_embeddings=True)
r_embs = model.encode([p[1] for p in sanity_pairs], normalize_embeddings=True)
u_embs = model.encode([p[2] for p in sanity_pairs], normalize_embeddings=True)
print("Semantic sanity check (gap must be positive):")
for i, (q, r_text, u_text) in enumerate(sanity_pairs):
    sim_r = float(cosine_similarity([q_embs[i]], [r_embs[i]])[0][0])
    sim_u = float(cosine_similarity([q_embs[i]], [u_embs[i]])[0][0])
    gap = sim_r - sim_u
    status = "✓" if gap > 0.05 else "✗ FAIL"
    print(f"  {status} Query {i+1}: related={sim_r:.3f} unrelated={sim_u:.3f} gap={gap:+.3f}")